# Código

In [7]:
import numpy as np
from sklearn.base import clone
from sklearn.pipeline import Pipeline
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "colab"


# ============================================================
# Helpers
# ============================================================
def _as_2d(X):
    X = np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return X


def _as_1d(y):
    return np.asarray(y).ravel()


def _first_not_none(*vals):
    for v in vals:
        if v is not None:
            return v
    return None


def _get_final_estimator(estimator):
    if isinstance(estimator, Pipeline):
        return estimator.steps[-1][1]
    return estimator


def _last_step_prefix(estimator):
    if isinstance(estimator, Pipeline):
        return estimator.steps[-1][0]
    return None


def _try_set_params(estimator, **params):
    try:
        estimator.set_params(**params)
        return True
    except Exception:
        return False


def _ema_smooth(arr, beta=0.85):
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return arr
    out = np.empty_like(arr, dtype=float)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = beta * out[i - 1] + (1 - beta) * arr[i]
    return out


def _sigmoid(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-z))


def _softmax(Z):
    Z = np.asarray(Z, dtype=float)
    Z = Z - np.max(Z, axis=1, keepdims=True)
    expZ = np.exp(Z)
    return expZ / np.sum(expZ, axis=1, keepdims=True)


def _one_hot(y, classes):
    y = _as_1d(y)
    classes = np.asarray(classes)
    idx = np.searchsorted(classes, y)
    Y = np.zeros((len(y), len(classes)), dtype=float)
    Y[np.arange(len(y)), idx] = 1.0
    return Y


def _binary_log_loss_from_p(p, y, eps=1e-12):
    p = np.clip(np.asarray(p, dtype=float).ravel(), eps, 1.0 - eps)
    y = _as_1d(y).astype(float)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


def _multiclass_cross_entropy(P, y, classes, eps=1e-12):
    P = np.clip(np.asarray(P, dtype=float), eps, 1.0)
    Y = _one_hot(y, classes)
    return float(-np.mean(np.sum(Y * np.log(P), axis=1)))


def _is_iterative(estimator):
    last = _get_final_estimator(estimator)
    return hasattr(last, "partial_fit") or hasattr(last, "warm_start")


def _find_standard_scaler(estimator):
    if not isinstance(estimator, Pipeline):
        return None

    for _, step in estimator.steps:
        has_transform = hasattr(step, "transform")
        has_mean = hasattr(step, "mean_")
        has_scale = hasattr(step, "scale_") or hasattr(step, "var_")
        if has_transform and has_mean and has_scale:
            return step
    return None


def _safe_get_scale(scaler):
    if scaler is None:
        return None, None, True, True

    mu = getattr(scaler, "mean_", None)

    scale = getattr(scaler, "scale_", None)
    if scale is None:
        var = getattr(scaler, "var_", None)
        if var is not None:
            scale = np.sqrt(np.asarray(var, dtype=float))
        else:
            scale = None

    with_mean = bool(getattr(scaler, "with_mean", True))
    with_std = bool(getattr(scaler, "with_std", True))

    return mu, scale, with_mean, with_std


def _theta_scaled_to_original_binary(w_s, b_s, scaler):
    w_s = np.asarray(w_s, dtype=float).ravel()
    b_s = float(np.asarray(b_s, dtype=float).ravel()[0]) if np.size(b_s) else float(b_s)

    if scaler is None:
        return w_s.copy(), float(b_s)

    mu, scale, with_mean, with_std = _safe_get_scale(scaler)
    dloc = w_s.size

    if (not with_std) or (scale is None):
        scale = np.ones(dloc, dtype=float)
    else:
        scale = np.asarray(scale, dtype=float).ravel()
        if scale.size != dloc:
            raise ValueError(f"Scaler/coef mismatch: scale has {scale.size}, coef has {dloc}.")

    if (not with_mean) or (mu is None):
        mu = np.zeros(dloc, dtype=float)
    else:
        mu = np.asarray(mu, dtype=float).ravel()
        if mu.size != dloc:
            raise ValueError(f"Scaler/coef mismatch: mean has {mu.size}, coef has {dloc}.")

    denom = scale + 1e-12
    w_o = w_s / denom
    b_o = float(b_s - np.sum(w_s * mu / denom))
    return w_o, b_o


def _theta_scaled_to_original_multiclass(W_s, b_s, scaler):
    W_s = np.asarray(W_s, dtype=float)  # (d, K)
    b_s = np.asarray(b_s, dtype=float).ravel()  # (K,)

    if scaler is None:
        return W_s.copy(), b_s.copy()

    mu, scale, with_mean, with_std = _safe_get_scale(scaler)
    dloc, K = W_s.shape

    if (not with_std) or (scale is None):
        scale = np.ones(dloc, dtype=float)
    else:
        scale = np.asarray(scale, dtype=float).ravel()
        if scale.size != dloc:
            raise ValueError(f"Scaler/coef mismatch: scale has {scale.size}, coef has {dloc}.")

    if (not with_mean) or (mu is None):
        mu = np.zeros(dloc, dtype=float)
    else:
        mu = np.asarray(mu, dtype=float).ravel()
        if mu.size != dloc:
            raise ValueError(f"Scaler/coef mismatch: mean has {mu.size}, coef has {dloc}.")

    denom = (scale + 1e-12)[:, None]
    W_o = W_s / denom
    b_o = b_s - np.sum(W_s * (mu[:, None] / denom), axis=0)
    return W_o, b_o


def _transform_up_to_last(pipeline, X):
    Xt = X
    for _, step in pipeline.steps[:-1]:
        if hasattr(step, "transform"):
            Xt = step.transform(Xt)
    return Xt


def _make_iterative_replay_estimator(estimator):
    est = clone(estimator)

    pref = _last_step_prefix(est)

    def p(name):
        return f"{pref}__{name}" if pref is not None else name

    _try_set_params(est, **{p("warm_start"): True})
    _try_set_params(est, **{p("max_iter"): 1})
    _try_set_params(est, **{p("tol"): None})
    _try_set_params(est, **{p("shuffle"): False})
    return est


def _get_classes_from_estimator_or_y(estimator, y):
    last = _get_final_estimator(estimator)
    if hasattr(last, "classes_"):
        return np.asarray(last.classes_)
    return np.unique(_as_1d(y))


def _predict_proba_any(estimator, X, classes=None):
    X = _as_2d(X)
    last = _get_final_estimator(estimator)

    if hasattr(estimator, "predict_proba"):
        P = estimator.predict_proba(X)
        return np.asarray(P, dtype=float)

    if hasattr(last, "predict_proba"):
        P = last.predict_proba(X)
        return np.asarray(P, dtype=float)

    if hasattr(estimator, "decision_function"):
        S = estimator.decision_function(X)
    elif hasattr(last, "decision_function"):
        S = last.decision_function(X)
    else:
        preds = estimator.predict(X)
        preds = np.asarray(preds).ravel()
        classes = np.unique(preds) if classes is None else np.asarray(classes)
        K = len(classes)
        if K == 2:
            p1 = (preds == classes[1]).astype(float)
            return np.column_stack([1.0 - p1, p1])
        Y = np.zeros((len(preds), K), dtype=float)
        idx = np.searchsorted(classes, preds)
        Y[np.arange(len(preds)), idx] = 1.0
        return Y

    S = np.asarray(S, dtype=float)
    if S.ndim == 1:
        p1 = _sigmoid(S)
        return np.column_stack([1.0 - p1, p1])

    return _softmax(S)


def _extract_theta_logistic_as_learned(estimator, d_expected=None):
    est = _get_final_estimator(estimator)

    if not (hasattr(est, "coef_") and hasattr(est, "intercept_")):
        return None

    W = np.asarray(est.coef_, dtype=float)
    b = np.asarray(est.intercept_, dtype=float).ravel()

    if W.ndim == 1:
        W = W.reshape(1, -1)

    # binary sklearn often stores coef shape (1, d), intercept shape (1,)
    K_eff, d = W.shape

    if d_expected is not None and d != int(d_expected):
        return None

    if K_eff == 1:
        return {
            "task": "binary",
            "w": W[0].copy(),
            "b": float(b[0]) if b.size else 0.0,
        }

    # multiclass
    return {
        "task": "multiclass",
        "W": W.T.copy(),   # return as (d, K) to match display conventions
        "b": b.copy(),     # (K,)
    }


def _convert_theta_for_display(theta_obj, scaler, display_space):
    if theta_obj is None:
        return None

    if display_space == "scaled" or scaler is None:
        return theta_obj

    if theta_obj["task"] == "binary":
        w_o, b_o = _theta_scaled_to_original_binary(theta_obj["w"], theta_obj["b"], scaler)
        return {"task": "binary", "w": w_o, "b": b_o}

    W_o, b_o = _theta_scaled_to_original_multiclass(theta_obj["W"], theta_obj["b"], scaler)
    return {"task": "multiclass", "W": W_o, "b": b_o}


def _empirical_prior_binary(y):
    y = _as_1d(y).astype(float)
    p1 = float(np.mean(y))
    return np.column_stack([np.full_like(y, 1.0 - p1, dtype=float), np.full_like(y, p1, dtype=float)])


def _empirical_prior_multiclass(y, classes):
    y = _as_1d(y)
    classes = np.asarray(classes)
    counts = np.array([(y == c).mean() for c in classes], dtype=float)
    return np.tile(counts.reshape(1, -1), (len(y), 1))


def _grid_prior_binary(size, p1):
    return np.full(size, float(p1), dtype=float)


def _grid_prior_multiclass(size, probs):
    probs = np.asarray(probs, dtype=float).ravel()
    return np.tile(probs.reshape(1, -1), (size, 1))


# ============================================================
# Main: capture predictive history (LOGISTIC)
# ============================================================
def fit_history_logistic(
    trained_estimator,
    X,
    y,
    *,
    steps=60,
    mode="auto",                 # "auto" | "iterative" | "final_interp"
    smooth=None,                 # None | "ema"
    smooth_beta=0.85,
    grid_1d_points=300,
    grid_2d_points=40,
    baseline="prior",            # "prior" | "uniform"
    display_space="original",    # "original" | "scaled"
):
    """
    Historia robusta para regresión logística binaria o multiclase.

    Binary:
      - d=1 -> p_line_hist
      - d=2 -> p_plane_hist
      - d>2 -> theta/loss only

    Multiclass:
      - d=1 -> p_curves_hist
      - d>1 -> theta/loss only

    Returns dict with:
      history_kind
      classes
      is_multiclass
      loss_hist
      grid
      p_line_hist / p_plane_hist / p_curves_hist
      w_hist / b_hist (binary or multiclass display-ready)
      w_hist_learned / b_hist_learned
      display_space
    """
    if steps < 1:
        raise ValueError("steps must be >= 1.")
    if display_space not in ("original", "scaled"):
        raise ValueError("display_space must be 'original' or 'scaled'.")
    if baseline not in ("prior", "uniform"):
        raise ValueError("baseline must be 'prior' or 'uniform'.")
    if mode not in ("auto", "iterative", "final_interp"):
        raise ValueError("mode must be 'auto', 'iterative', or 'final_interp'.")

    X = _as_2d(X)
    y = _as_1d(y)
    n, d = X.shape

    classes = _get_classes_from_estimator_or_y(trained_estimator, y)
    K = len(classes)
    is_multiclass = K > 2

    scaler_trained = _find_standard_scaler(trained_estimator)

    grid = {}
    x1_grid = None
    X1g = X2g = None

    if d == 1:
        x1 = X[:, 0]
        x1_grid = np.linspace(float(x1.min()), float(x1.max()), int(grid_1d_points))
        grid["x1_grid"] = x1_grid

    elif d == 2:
        x1 = X[:, 0]
        x2 = X[:, 1]
        x1_grid = np.linspace(float(x1.min()), float(x1.max()), int(grid_2d_points))
        x2_grid = np.linspace(float(x2.min()), float(x2.max()), int(grid_2d_points))
        X1g, X2g = np.meshgrid(x1_grid, x2_grid)
        grid["x1_grid"] = x1_grid
        grid["x2_grid"] = x2_grid
        grid["X1g"] = X1g
        grid["X2g"] = X2g

    iterative_like = _is_iterative(trained_estimator)
    if mode == "auto":
        mode = "iterative" if iterative_like else "final_interp"

    loss_hist = np.zeros(steps, dtype=float)

    # theta histories
    w_hist = None
    b_hist = None
    w_hist_learned = None
    b_hist_learned = None

    # predictive histories
    p_line_hist = None        # binary d=1
    p_plane_hist = None       # binary d=2
    p_curves_hist = None      # multiclass d=1

    if (not is_multiclass) and d == 1 and x1_grid is not None:
        p_line_hist = np.zeros((steps, x1_grid.size), dtype=float)

    if (not is_multiclass) and d == 2 and X1g is not None:
        p_plane_hist = np.zeros((steps, X1g.shape[0], X1g.shape[1]), dtype=float)

    if is_multiclass and d == 1 and x1_grid is not None:
        p_curves_hist = np.zeros((steps, x1_grid.size, K), dtype=float)

    history_kind = None
    scaler_for_display = scaler_trained

    def _loss_from_P(P):
        if not is_multiclass:
            p1 = np.asarray(P[:, 1], dtype=float)
            y_bin = (y == classes[1]).astype(float)
            return _binary_log_loss_from_p(p1, y_bin)
        return _multiclass_cross_entropy(P, y, classes)

    def _pred_grid(est):
        if d == 1 and x1_grid is not None:
            Xg = x1_grid.reshape(-1, 1)
            Pg = _predict_proba_any(est, Xg, classes=classes)
            if is_multiclass:
                return Pg
            return Pg[:, 1]
        if d == 2 and X1g is not None and (not is_multiclass):
            Xg = np.column_stack([X1g.ravel(), X2g.ravel()])
            Pg = _predict_proba_any(est, Xg, classes=classes)
            return Pg[:, 1].reshape(X1g.shape)
        return None

    # ============================================================
    # A) ITERATIVE
    # ============================================================
    if mode == "iterative":
        if not iterative_like:
            mode = "final_interp"
        else:
            est_replay = _make_iterative_replay_estimator(trained_estimator)

            est_replay.fit(X, y)
            scaler_for_display = _find_standard_scaler(est_replay)

            theta0 = _extract_theta_logistic_as_learned(est_replay, d_expected=d)
            if theta0 is not None:
                if theta0["task"] == "binary":
                    w_hist = np.zeros((steps, d), dtype=float)
                    b_hist = np.zeros(steps, dtype=float)
                    w_hist[0] = theta0["w"]
                    b_hist[0] = theta0["b"]
                else:
                    w_hist = np.zeros((steps, d, K), dtype=float)
                    b_hist = np.zeros((steps, K), dtype=float)
                    w_hist[0] = theta0["W"]
                    b_hist[0] = theta0["b"]

            last = _get_final_estimator(est_replay)
            if hasattr(last, "t_"):
                try:
                    last.t_ = 1.0
                except Exception:
                    pass

            P0 = _predict_proba_any(est_replay, X, classes=classes)
            loss_hist[0] = _loss_from_P(P0)

            g0 = _pred_grid(est_replay)
            if p_line_hist is not None:
                p_line_hist[0] = np.asarray(g0, dtype=float)
            if p_plane_hist is not None:
                p_plane_hist[0] = np.asarray(g0, dtype=float)
            if p_curves_hist is not None:
                p_curves_hist[0] = np.asarray(g0, dtype=float)

            is_pipe = isinstance(est_replay, Pipeline)
            last_step_est = _get_final_estimator(est_replay)
            has_pf_last = hasattr(last_step_est, "partial_fit")

            can_pf_pipe = False
            Xt = None
            if is_pipe:
                try:
                    Xt = _transform_up_to_last(est_replay, X)
                    can_pf_pipe = hasattr(est_replay.steps[-1][1], "partial_fit")
                except Exception:
                    Xt = None
                    can_pf_pipe = False

            for t in range(1, steps):
                if (not is_pipe) and has_pf_last:
                    est_replay.partial_fit(X, y)

                elif is_pipe and can_pf_pipe and Xt is not None:
                    est_replay.steps[-1][1].partial_fit(Xt, y)

                else:
                    est_replay.fit(X, y)

                Pt = _predict_proba_any(est_replay, X, classes=classes)
                loss_hist[t] = _loss_from_P(Pt)

                gt = _pred_grid(est_replay)
                if p_line_hist is not None:
                    p_line_hist[t] = np.asarray(gt, dtype=float)
                if p_plane_hist is not None:
                    p_plane_hist[t] = np.asarray(gt, dtype=float)
                if p_curves_hist is not None:
                    p_curves_hist[t] = np.asarray(gt, dtype=float)

                if w_hist is not None:
                    thetat = _extract_theta_logistic_as_learned(est_replay, d_expected=d)
                    if thetat is not None:
                        if thetat["task"] == "binary":
                            w_hist[t] = thetat["w"]
                            b_hist[t] = thetat["b"]
                        else:
                            w_hist[t] = thetat["W"]
                            b_hist[t] = thetat["b"]

            history_kind = "iterative"

    # ============================================================
    # B) FINAL_INTERP
    # ============================================================
    if mode == "final_interp":
        PF = _predict_proba_any(trained_estimator, X, classes=classes)

        if baseline == "uniform":
            if not is_multiclass:
                P0 = np.column_stack([
                    np.full(n, 0.5, dtype=float),
                    np.full(n, 0.5, dtype=float),
                ])
            else:
                P0 = np.full((n, K), 1.0 / K, dtype=float)
        else:
            if not is_multiclass:
                y_bin = (y == classes[1]).astype(float)
                p1 = float(np.mean(y_bin))
                P0 = np.column_stack([
                    np.full(n, 1.0 - p1, dtype=float),
                    np.full(n, p1, dtype=float),
                ])
            else:
                priors = np.array([(y == c).mean() for c in classes], dtype=float)
                P0 = np.tile(priors.reshape(1, -1), (n, 1))

        g0 = None
        gF = None

        if d == 1 and x1_grid is not None:
            Xg = x1_grid.reshape(-1, 1)
            PgF = _predict_proba_any(trained_estimator, Xg, classes=classes)

            if not is_multiclass:
                if baseline == "uniform":
                    p0 = np.full(x1_grid.size, 0.5, dtype=float)
                else:
                    y_bin = (y == classes[1]).astype(float)
                    p0 = np.full(x1_grid.size, float(np.mean(y_bin)), dtype=float)

                g0 = p0
                gF = PgF[:, 1]
            else:
                if baseline == "uniform":
                    g0 = np.full((x1_grid.size, K), 1.0 / K, dtype=float)
                else:
                    priors = np.array([(y == c).mean() for c in classes], dtype=float)
                    g0 = np.tile(priors.reshape(1, -1), (x1_grid.size, 1))
                gF = PgF

        if d == 2 and X1g is not None and (not is_multiclass):
            Xg = np.column_stack([X1g.ravel(), X2g.ravel()])
            PgF = _predict_proba_any(trained_estimator, Xg, classes=classes)[:, 1].reshape(X1g.shape)

            if baseline == "uniform":
                g0 = np.full_like(X1g, 0.5, dtype=float)
            else:
                y_bin = (y == classes[1]).astype(float)
                g0 = np.full_like(X1g, float(np.mean(y_bin)), dtype=float)

            gF = PgF

        thetaF = _extract_theta_logistic_as_learned(trained_estimator, d_expected=d)
        if thetaF is not None:
            if thetaF["task"] == "binary":
                w_hist = np.tile(thetaF["w"].reshape(1, -1), (steps, 1))
                b_hist = np.full(steps, float(thetaF["b"]), dtype=float)
            else:
                w_hist = np.tile(thetaF["W"].reshape(1, d, K), (steps, 1, 1))
                b_hist = np.tile(thetaF["b"].reshape(1, K), (steps, 1))

        for t in range(steps):
            alpha = t / (steps - 1) if steps > 1 else 1.0
            Pt = (1.0 - alpha) * P0 + alpha * PF
            loss_hist[t] = _loss_from_P(Pt)

            if p_line_hist is not None and g0 is not None:
                p_line_hist[t] = (1.0 - alpha) * g0 + alpha * gF

            if p_plane_hist is not None and g0 is not None:
                p_plane_hist[t] = (1.0 - alpha) * g0 + alpha * gF

            if p_curves_hist is not None and g0 is not None:
                p_curves_hist[t] = (1.0 - alpha) * g0 + alpha * gF

        history_kind = "final_interp"

    # ============================================================
    # VISUAL smoothing only
    # ============================================================
    if smooth == "ema":
        loss_hist = _ema_smooth(loss_hist, beta=smooth_beta)

        if p_line_hist is not None:
            for j in range(p_line_hist.shape[1]):
                p_line_hist[:, j] = _ema_smooth(p_line_hist[:, j], beta=smooth_beta)

        if p_plane_hist is not None:
            Z = p_plane_hist.reshape(steps, -1)
            for j in range(Z.shape[1]):
                Z[:, j] = _ema_smooth(Z[:, j], beta=smooth_beta)
            p_plane_hist = Z.reshape(p_plane_hist.shape)

        if p_curves_hist is not None:
            P = p_curves_hist.reshape(steps, -1)
            for j in range(P.shape[1]):
                P[:, j] = _ema_smooth(P[:, j], beta=smooth_beta)
            p_curves_hist = P.reshape(p_curves_hist.shape)

    # ============================================================
    # theta display-space conversion
    # ============================================================
    if w_hist is not None and b_hist is not None:
        w_hist_learned = np.asarray(w_hist, dtype=float).copy()
        b_hist_learned = np.asarray(b_hist, dtype=float).copy()

        if display_space == "original" and scaler_for_display is not None:
            if not is_multiclass:
                w_show = np.zeros_like(w_hist_learned)
                b_show = np.zeros_like(b_hist_learned)
                for t in range(w_hist_learned.shape[0]):
                    wo, bo = _theta_scaled_to_original_binary(w_hist_learned[t], b_hist_learned[t], scaler_for_display)
                    w_show[t] = wo
                    b_show[t] = bo
                w_hist = w_show
                b_hist = b_show
            else:
                w_show = np.zeros_like(w_hist_learned)
                b_show = np.zeros_like(b_hist_learned)
                for t in range(w_hist_learned.shape[0]):
                    Wo, bo = _theta_scaled_to_original_multiclass(w_hist_learned[t], b_hist_learned[t], scaler_for_display)
                    w_show[t] = Wo
                    b_show[t] = bo
                w_hist = w_show
                b_hist = b_show

    return {
        "history_kind": history_kind,
        "classes": classes,
        "is_multiclass": is_multiclass,
        "loss_hist": loss_hist,
        "grid": grid,
        "p_line_hist": p_line_hist,
        "p_plane_hist": p_plane_hist,
        "p_curves_hist": p_curves_hist,
        "w_hist": w_hist,
        "b_hist": b_hist,
        "w_hist_learned": w_hist_learned,
        "b_hist_learned": b_hist_learned,
        "display_space": display_space,
    }


# ============================================================
# Binary logistic (1 variable)
# ============================================================
def build_binary_simple_logistic_figure(
    x1, y,
    w_hist=None, b_hist=None,
    *,
    p_line_hist=None,
    x1_grid=None,
    loss_hist=None,
    show_loss=False,
    history_kind="iterative",
    title="Binary Logistic Regression (1 variable)",
    strict_loss=False,
    dec=4,
    jitter=0.03,
):
    if show_loss and history_kind != "iterative":
        if strict_loss:
            raise ValueError("show_loss=True is only allowed for iterative histories.")
        show_loss = False
        loss_hist = None

    x1 = np.asarray(x1).ravel()
    y = np.asarray(y).ravel().astype(float)
    y_jitter = y + np.random.uniform(-jitter, jitter, size=y.size)

    use_pred_grid = p_line_hist is not None

    if use_pred_grid:
        p_line_hist = np.asarray(p_line_hist, dtype=float)
        if x1_grid is None:
            raise ValueError("If p_line_hist is provided, x1_grid must be provided.")
        x1_grid = np.asarray(x1_grid, dtype=float).ravel()

        if p_line_hist.ndim != 2:
            raise ValueError("p_line_hist must have shape (steps, grid_points).")
        if p_line_hist.shape[1] != x1_grid.size:
            raise ValueError("p_line_hist second dim must match x1_grid size.")

        steps_n = int(p_line_hist.shape[0])

        def p_line(t):
            return p_line_hist[t]

        w_disp = None
        b_disp = None
        if w_hist is not None and b_hist is not None:
            w_arr = np.asarray(w_hist, dtype=float)
            b_arr = np.asarray(b_hist, dtype=float).ravel()
            if w_arr.ndim == 2 and w_arr.shape[1] == 1:
                w_arr = w_arr[:, 0]
            if w_arr.ndim == 1 and w_arr.size == steps_n and b_arr.size == steps_n:
                w_disp = w_arr
                b_disp = b_arr

        def formula_text():
            return r"$\hat{p}(y=1\mid x)=\sigma(z),\;\; z=\theta_1x_1+\theta_0$"

        def eq_text(t):
            if w_disp is None:
                return r"$\sigma(z)=\dfrac{1}{1+e^{-z}},\;\;\hat{p}(y=1\mid x)=f(x_1)$"
            return (
                r"$\sigma(z)=\dfrac{1}{1+e^{-z}}"
                + rf",\;\; z=({w_disp[t]:.{dec}f})x_1+({b_disp[t]:.{dec}f})$"
            )

        x_min, x_max = float(x1_grid.min()), float(x1_grid.max())

    else:
        if w_hist is None or b_hist is None:
            raise ValueError("Legacy mode requires w_hist and b_hist. Prefer p_line_hist + x1_grid.")

        w_hist = np.asarray(w_hist, dtype=float)
        b_hist = np.asarray(b_hist, dtype=float).ravel()
        steps_n = int(b_hist.size)

        if w_hist.ndim == 1:
            w_hist = w_hist.reshape(-1, 1)
        if w_hist.shape[0] != steps_n:
            raise ValueError("w_hist and b_hist must have same number of steps.")
        if w_hist.shape[1] != 1:
            raise ValueError(f"Binary 1D logistic expects 1 weight, got d={w_hist.shape[1]}.")

        x_min, x_max = float(x1.min()), float(x1.max())
        x1_grid = np.linspace(x_min, x_max, 300)

        def p_line(t):
            w1 = float(w_hist[t, 0])
            b = float(b_hist[t])
            return _sigmoid(w1 * x1_grid + b)

        def formula_text():
            return r"$\hat{p}(y=1\mid x)=\sigma(z),\;\; z=\theta_1x_1+\theta_0$"

        def eq_text(t):
            w1 = float(w_hist[t, 0])
            b = float(b_hist[t])
            return (
                r"$\sigma(z)=\dfrac{1}{1+e^{-z}}"
                + rf",\;\; z=({w1:.{dec}f})x_1+({b:.{dec}f})$"
            )

    if steps_n < 1:
        raise ValueError("Need at least 1 step to animate.")

    if show_loss:
        if loss_hist is None:
            raise ValueError("show_loss=True requires loss_hist.")
        loss_hist = np.asarray(loss_hist, dtype=float).ravel()
        if loss_hist.size != steps_n:
            raise ValueError("loss_hist must have same length as steps.")

    step_axis = np.arange(steps_n)

    if show_loss:
        theta_y = 1.18
        eq_y = 1.10
        margin_t = 160
    else:
        theta_y = 1.15
        eq_y = 1.05
        margin_t = 150

    def formula_annotation():
        return dict(
            x=0.5, y=theta_y,
            xref="paper", yref="paper",
            text=formula_text(),
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(color="white", size=16),
        )

    def eq_annotation(t):
        return dict(
            x=0.5, y=eq_y,
            xref="paper", yref="paper",
            text=eq_text(t),
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(color="white", size=16),
        )

    def _pad(lo, hi, frac=0.10):
        span = (hi - lo) + 1e-9
        return [lo - frac * span, hi + frac * span]

    x_range = _pad(x_min, x_max)

    if show_loss:
        lmin, lmax = float(loss_hist.min()), float(loss_hist.max())
        lpad = 0.10 * (lmax - lmin + 1e-9)

    if show_loss:
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.62, 0.38],
            horizontal_spacing=0.08,
            specs=[[{"type": "xy"}, {"type": "xy"}]],
        )

        fig.add_trace(
            go.Scatter(
                x=x1, y=y_jitter,
                mode="markers",
                name="Data",
                marker=dict(size=7, opacity=0.80),
                legendgroup="fit",
                showlegend=True,
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=x1_grid, y=p_line(0),
                mode="lines",
                name="Model",
                line=dict(width=4),
                legendgroup="fit",
                showlegend=True,
                uid="MODEL_LINE",
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=[0], y=[loss_hist[0]],
                mode="lines",
                name="Log-loss",
                line=dict(width=3),
                legendgroup="loss",
                showlegend=True,
                uid="LOSS_LINE",
            ),
            row=1, col=2
        )

        frames = []
        for t in range(steps_n):
            frames.append(
                go.Frame(
                    name=str(t),
                    data=[
                        go.Scatter(x=x1_grid, y=p_line(t), mode="lines", line=dict(width=4), uid="MODEL_LINE"),
                        go.Scatter(x=step_axis[:t + 1], y=loss_hist[:t + 1], mode="lines", line=dict(width=3), uid="LOSS_LINE"),
                    ],
                    traces=[1, 2],
                    layout=go.Layout(annotations=[formula_annotation(), eq_annotation(t)]),
                )
            )
        fig.frames = frames

        fig.update_layout(
            template="plotly_dark",
            height=720,
            font=dict(family="Helvetica", color="white"),
            title=dict(
                text=title,
                y=0.96,
                x=0.5,
                xanchor="center",
                font=dict(color="white", size=24),
            ),
            annotations=[formula_annotation(), eq_annotation(0)],
            margin=dict(t=margin_t, r=30, l=60, b=70),
            legend=dict(
                orientation="v",
                x=0.49, y=0.02,
                xanchor="right",
                yanchor="bottom",
                bgcolor="rgba(220,220,220,0.85)",
                bordercolor="rgba(0,0,0,0.6)",
                borderwidth=1,
                font=dict(size=12, color="black"),
            ),
            legend2=dict(
                orientation="v",
                x=0.985, y=0.02,
                xanchor="right",
                yanchor="bottom",
                bgcolor="rgba(220,220,220,0.85)",
                bordercolor="rgba(0,0,0,0.6)",
                borderwidth=1,
                font=dict(size=12, color="black"),
            ),
            sliders=[dict(
                active=0,
                currentvalue=dict(prefix="Step: "),
                pad=dict(t=55),
                steps=[
                    dict(
                        method="animate",
                        args=[[str(t)], {
                            "mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0},
                        }],
                        label=str(t),
                    ) for t in range(steps_n)
                ],
            )],
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0.10,
                y=1.14,
                bgcolor="white",
                bordercolor="black",
                borderwidth=1,
                font=dict(color="black", size=14),
                buttons=[
                    dict(label="Play", method="animate",
                         args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate",
                         args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )],
        )

        fig.data[2].update(legend="legend2")
        fig.update_xaxes(title="x₁", range=x_range, row=1, col=1)
        fig.update_yaxes(title=r"$\hat{p}(y=1\mid x)$", range=[-0.08, 1.08], row=1, col=1)
        fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=2)
        fig.update_yaxes(title="Log-loss", range=[lmin - lpad, lmax + lpad], row=1, col=2)
        return fig

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=x1, y=y_jitter,
            mode="markers",
            name="Data",
            marker=dict(size=7, opacity=0.80),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=x1_grid, y=p_line(0),
            mode="lines",
            name="Model",
            line=dict(width=4),
            uid="MODEL_LINE",
        )
    )

    frames = []
    for t in range(steps_n):
        frames.append(
            go.Frame(
                name=str(t),
                data=[go.Scatter(x=x1_grid, y=p_line(t), mode="lines", line=dict(width=4), uid="MODEL_LINE")],
                traces=[1],
                layout=go.Layout(annotations=[formula_annotation(), eq_annotation(t)]),
            )
        )
    fig.frames = frames

    fig.update_layout(
        template="plotly_dark",
        height=720,
        font=dict(family="Helvetica", color="white"),
        title=dict(
            text=title,
            y=0.96,
            x=0.5,
            xanchor="center",
            font=dict(color="white", size=24),
        ),
        annotations=[formula_annotation(), eq_annotation(0)],
        margin=dict(t=margin_t, r=60, l=70, b=80),
        legend=dict(
            x=0.985, y=0.02,
            xanchor="right", yanchor="bottom",
            bgcolor="rgba(220,220,220,0.85)",
            bordercolor="rgba(0,0,0,0.6)",
            borderwidth=1,
            font=dict(color="black", size=12),
        ),
        xaxis=dict(title="x₁", range=x_range),
        yaxis=dict(title=r"$\hat{p}(y=1\mid x)$", range=[-0.08, 1.08]),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=55),
            steps=[
                dict(
                    method="animate",
                    args=[[str(t)], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0},
                    }],
                    label=str(t),
                ) for t in range(steps_n)
            ],
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.10,
            y=1.14,
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(color="black", size=14),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
    )

    return fig


# ============================================================
# Binary logistic (2 variables)
# ============================================================
def build_binary_plane_logistic_figure(
    x1, x2, y,
    w_hist=None, b_hist=None,
    *,
    p_plane_hist=None,
    X1g=None, X2g=None,
    loss_hist=None,
    show_loss=False,
    history_kind="iterative",
    title="Binary Logistic Regression (2 variables)",
    strict_loss=False,
    dec=4,
    jitter=0.03,
):
    if show_loss and history_kind != "iterative":
        if strict_loss:
            raise ValueError("show_loss=True is only allowed for iterative histories.")
        show_loss = False
        loss_hist = None

    x1 = np.asarray(x1).ravel()
    x2 = np.asarray(x2).ravel()
    y = np.asarray(y).ravel().astype(float)
    y_jitter = y + np.random.uniform(-jitter, jitter, size=y.size)

    use_pred_grid = p_plane_hist is not None

    if use_pred_grid:
        p_plane_hist = np.asarray(p_plane_hist, dtype=float)
        if X1g is None or X2g is None:
            raise ValueError("If p_plane_hist is provided, X1g and X2g must be provided.")

        X1g = np.asarray(X1g, dtype=float)
        X2g = np.asarray(X2g, dtype=float)

        if X1g.shape != X2g.shape:
            raise ValueError("X1g and X2g must have same shape.")
        if p_plane_hist.ndim != 3:
            raise ValueError("p_plane_hist must have shape (steps, H, W).")
        if p_plane_hist.shape[1:] != X1g.shape:
            raise ValueError("p_plane_hist grid shape must match X1g/X2g.")

        steps_n = int(p_plane_hist.shape[0])

        def p_plane(t):
            return p_plane_hist[t]

        w_disp = None
        b_disp = None
        if w_hist is not None and b_hist is not None:
            w_arr = np.asarray(w_hist, dtype=float)
            b_arr = np.asarray(b_hist, dtype=float).ravel()
            if w_arr.ndim == 2 and w_arr.shape == (steps_n, 2) and b_arr.size == steps_n:
                w_disp = w_arr
                b_disp = b_arr

        def formula_text():
            return r"$\hat{p}(y=1\mid \mathbf{x})=\sigma(z),\;\; z=\theta_1x_1+\theta_2x_2+\theta_0$"

        def eq_text(t):
            if w_disp is None:
                return r"$\hat{p}(y=1\mid \mathbf{x})=f(x_1,x_2)$"
            w1 = float(w_disp[t, 0])
            w2 = float(w_disp[t, 1])
            b = float(b_disp[t])
            return (
                r"$\sigma(z)=\dfrac{1}{1+e^{-z}}"
                + rf",\;\; z=({w1:.{dec}f})x_1+({w2:.{dec}f})x_2+({b:.{dec}f})$"
            )

        x1_min, x1_max = float(np.min(X1g)), float(np.max(X1g))
        x2_min, x2_max = float(np.min(X2g)), float(np.max(X2g))

    else:
        if w_hist is None or b_hist is None:
            raise ValueError("Legacy mode requires w_hist and b_hist. Prefer p_plane_hist + X1g/X2g.")

        w_hist = np.asarray(w_hist, dtype=float)
        b_hist = np.asarray(b_hist, dtype=float).ravel()
        steps_n = int(b_hist.size)

        if w_hist.ndim == 1:
            if w_hist.size == steps_n * 2:
                w_hist = w_hist.reshape(steps_n, 2)
            else:
                raise ValueError("Expected w_hist shape (steps, 2).")
        if w_hist.ndim != 2 or w_hist.shape != (steps_n, 2):
            raise ValueError("Expected w_hist shape (steps, 2) and b_hist shape (steps,).")

        x1_grid = np.linspace(float(x1.min()), float(x1.max()), 40)
        x2_grid = np.linspace(float(x2.min()), float(x2.max()), 40)
        X1g, X2g = np.meshgrid(x1_grid, x2_grid)

        def p_plane(t):
            w1 = float(w_hist[t, 0])
            w2 = float(w_hist[t, 1])
            b = float(b_hist[t])
            return _sigmoid(w1 * X1g + w2 * X2g + b)

        def formula_text():
            return r"$\hat{p}(y=1\mid \mathbf{x})=\sigma(z),\;\; z=\theta_1x_1+\theta_2x_2+\theta_0$"

        def eq_text(t):
            w1 = float(w_hist[t, 0])
            w2 = float(w_hist[t, 1])
            b = float(b_hist[t])
            return (
                r"$\sigma(z)=\dfrac{1}{1+e^{-z}}"
                + rf",\;\; z=({w1:.{dec}f})x_1+({w2:.{dec}f})x_2+({b:.{dec}f})$"
            )

        x1_min, x1_max = float(np.min(X1g)), float(np.max(X1g))
        x2_min, x2_max = float(np.min(X2g)), float(np.max(X2g))

    if steps_n < 1:
        raise ValueError("Need at least 1 step to animate.")

    if show_loss:
        if loss_hist is None:
            raise ValueError("show_loss=True requires loss_hist.")
        loss_hist = np.asarray(loss_hist, dtype=float).ravel()
        if loss_hist.size != steps_n:
            raise ValueError("loss_hist must match steps.")

    step_axis = np.arange(steps_n)

    if show_loss:
        theta_y = 1.16
        eq_y = 1.08
        margin_t = 150
    else:
        theta_y = 1.15
        eq_y = 1.05
        margin_t = 150

    def formula_annotation():
        return dict(
            x=0.5, y=theta_y,
            xref="paper", yref="paper",
            text=formula_text(),
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(color="white", size=16),
        )

    def eq_annotation(t):
        return dict(
            x=0.5, y=eq_y,
            xref="paper", yref="paper",
            text=eq_text(t),
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(color="white", size=16),
        )

    z0 = np.asarray(p_plane(0), dtype=float)
    zL = np.asarray(p_plane(steps_n - 1), dtype=float)
    z_all = np.concatenate([y_jitter, z0.ravel(), zL.ravel()])
    z_min, z_max = float(z_all.min()), float(z_all.max())

    def _pad(lo, hi, frac=0.10):
        span = (hi - lo) + 1e-9
        return [lo - frac * span, hi + frac * span]

    x1_range = _pad(x1_min, x1_max)
    x2_range = _pad(x2_min, x2_max)
    y_range = [min(-0.08, z_min - 0.03), max(1.08, z_max + 0.03)]

    CAMERA = dict(eye=dict(x=1.55, y=1.55, z=1.15))

    if show_loss:
        lmin, lmax = float(loss_hist.min()), float(loss_hist.max())
        lpad = 0.10 * (lmax - lmin + 1e-9)

    if show_loss:
        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.60, 0.30],
            horizontal_spacing=0.06,
            specs=[[{"type": "scene"}, {"type": "xy"}]],
        )

        fig.add_trace(
            go.Scatter3d(
                x=x1, y=x2, z=y_jitter,
                mode="markers",
                name="Data",
                marker=dict(size=4, opacity=0.85),
                legendgroup="fit",
                showlegend=True,
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Surface(
                x=X1g, y=X2g, z=p_plane(0),
                name="Model",
                opacity=0.55,
                showscale=False,
                legendgroup="fit",
                showlegend=True,
                uid="MODEL_SURFACE",
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=[0], y=[loss_hist[0]],
                mode="lines",
                name="Log-loss",
                line=dict(width=3),
                legendgroup="loss",
                showlegend=True,
                uid="LOSS_LINE",
            ),
            row=1, col=2
        )

        frames = []
        for t in range(steps_n):
            frames.append(
                go.Frame(
                    name=str(t),
                    data=[
                        go.Surface(
                            x=X1g, y=X2g, z=p_plane(t),
                            opacity=0.55,
                            showscale=False,
                            showlegend=True,
                            uid="MODEL_SURFACE",
                        ),
                        go.Scatter(
                            x=step_axis[:t + 1], y=loss_hist[:t + 1],
                            mode="lines",
                            line=dict(width=3),
                            uid="LOSS_LINE",
                        ),
                    ],
                    traces=[1, 2],
                    layout=go.Layout(
                        annotations=[formula_annotation(), eq_annotation(t)],
                        scene=dict(camera=CAMERA),
                    ),
                )
            )
        fig.frames = frames

        fig.update_layout(
            template="plotly_dark",
            font=dict(family="Helvetica", color="white"),
            height=720,
            title=dict(
                text=title,
                x=0.5, y=0.96,
                xanchor="center",
                font=dict(color="white", size=24),
            ),
            annotations=[formula_annotation(), eq_annotation(0)],
            margin=dict(t=margin_t, r=50, l=60, b=70),
            legend=dict(
                x=0.585, y=0.82,
                xanchor="right", yanchor="bottom",
                bgcolor="rgba(220,220,220,0.85)",
                bordercolor="rgba(0,0,0,0.6)",
                borderwidth=1,
                font=dict(color="black", size=12),
            ),
            legend2=dict(
                x=0.995, y=0.82,
                xanchor="right", yanchor="bottom",
                bgcolor="rgba(220,220,220,0.85)",
                bordercolor="rgba(0,0,0,0.6)",
                borderwidth=1,
                font=dict(color="black", size=12),
            ),
            scene=dict(
                xaxis=dict(title="x₁", range=x1_range),
                yaxis=dict(title="x₂", range=x2_range),
                zaxis=dict(title=r"$\hat{p}(y=1\mid \mathbf{x})$", range=y_range),
                aspectmode="cube",
                camera=CAMERA,
            ),
            sliders=[dict(
                active=0,
                currentvalue=dict(prefix="Step: "),
                pad=dict(t=55),
                steps=[
                    dict(
                        method="animate",
                        args=[[str(t)], {
                            "mode": "immediate",
                            "frame": {"duration": 0, "redraw": True},
                            "transition": {"duration": 0},
                        }],
                        label=str(t),
                    ) for t in range(steps_n)
                ],
            )],
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0.10,
                y=1.14,
                bgcolor="white",
                bordercolor="black",
                borderwidth=1,
                font=dict(color="black", size=14),
                buttons=[
                    dict(label="Play", method="animate",
                         args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate",
                         args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )],
        )

        fig.data[2].update(legend="legend2")
        fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=2)
        fig.update_yaxes(title="Log-loss", range=[lmin - lpad, lmax + lpad], row=1, col=2)
        return fig

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=x1, y=x2, z=y_jitter,
            mode="markers",
            name="Data",
            marker=dict(size=4, opacity=0.85),
        )
    )

    fig.add_trace(
        go.Surface(
            x=X1g, y=X2g, z=p_plane(0),
            name="Model",
            opacity=0.55,
            showscale=False,
            showlegend=True,
            uid="MODEL_SURFACE",
        )
    )

    frames = []
    for t in range(steps_n):
        frames.append(
            go.Frame(
                name=str(t),
                data=[go.Surface(
                    x=X1g, y=X2g, z=p_plane(t),
                    opacity=0.55,
                    showscale=False,
                    showlegend=True,
                    uid="MODEL_SURFACE",
                )],
                traces=[1],
                layout=go.Layout(
                    annotations=[formula_annotation(), eq_annotation(t)],
                    scene=dict(camera=CAMERA),
                ),
            )
        )
    fig.frames = frames

    fig.update_layout(
        template="plotly_dark",
        height=720,
        font=dict(family="Helvetica", color="white"),
        title=dict(
            text=title,
            y=0.96,
            x=0.5,
            xanchor="center",
            font=dict(color="white", size=24),
        ),
        annotations=[formula_annotation(), eq_annotation(0)],
        margin=dict(t=margin_t, r=30, l=60, b=70),
        showlegend=True,
        legend=dict(
            x=0.985, y=0.02,
            xanchor="right", yanchor="bottom",
            bgcolor="rgba(220,220,220,0.85)",
            bordercolor="rgba(0,0,0,0.6)",
            borderwidth=1,
            font=dict(color="black", size=12),
        ),
        scene=dict(
            xaxis=dict(title="x₁", range=x1_range),
            yaxis=dict(title="x₂", range=x2_range),
            zaxis=dict(title=r"$\hat{p}(y=1\mid \mathbf{x})$", range=y_range),
            aspectmode="cube",
            camera=CAMERA,
        ),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=55),
            steps=[
                dict(
                    method="animate",
                    args=[[str(t)], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0},
                    }],
                    label=str(t),
                ) for t in range(steps_n)
            ],
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.10,
            y=1.14,
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(color="black", size=14),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
    )

    return fig


# ============================================================
# Binary logistic multivariable (d > 2)
# ============================================================
def build_binary_multivar_logistic_figure(
    X, y,
    w_hist, b_hist,
    *,
    loss_hist=None,
    show_loss=True,
    history_kind="iterative",
    title=None,
    strict_loss=False,
    terms_per_line=6,
    dec=4,
    threshold_dense=100,
):
    if show_loss and history_kind != "iterative":
        if strict_loss:
            raise ValueError("show_loss=True is only allowed for iterative histories.")
        show_loss = False
        loss_hist = None

    X = np.asarray(X)
    y = np.asarray(y).ravel()
    w_hist = np.asarray(w_hist, dtype=float)
    b_hist = np.asarray(b_hist, dtype=float).ravel()

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if w_hist.ndim == 1:
        steps_n = int(b_hist.size)
        if steps_n < 1:
            raise ValueError("Need at least 1 step to animate.")
        if w_hist.size % steps_n != 0:
            raise ValueError("w_hist cannot be reshaped to (steps, d).")
        d = int(w_hist.size // steps_n)
        w_hist = w_hist.reshape(steps_n, d)
    else:
        steps_n = int(b_hist.size)
        if steps_n < 1:
            raise ValueError("Need at least 1 step to animate.")
        if w_hist.ndim != 2:
            raise ValueError("w_hist must have shape (steps, d).")
        if w_hist.shape[0] != steps_n:
            raise ValueError("w_hist and b_hist must match in steps.")
        d = int(w_hist.shape[1])

    d_X = int(X.shape[1])
    if d_X != d:
        raise ValueError(f"X has d={d_X} but w_hist has d={d}.")

    if d <= 2:
        raise ValueError("This figure is intended for d > 2.")

    if title is None:
        title = f"Binary Logistic Regression ({d} variables)"

    if show_loss:
        if loss_hist is None:
            raise ValueError("show_loss=True requires loss_hist.")
        loss_hist = np.asarray(loss_hist, dtype=float).ravel()
        if loss_hist.size != steps_n:
            raise ValueError("loss_hist must have same length as b_hist.")

    step_axis = np.arange(steps_n)

    if show_loss:
        lmin, lmax = float(loss_hist.min()), float(loss_hist.max())
        lpad = 0.08 * ((lmax - lmin) + 1e-9)

    def _needs_single_col(values, max_digits=5):
        vals = np.asarray(values, dtype=float).ravel()
        for v in vals:
            if not np.isfinite(v):
                return True
            int_digits = len(str(int(abs(float(v)))))
            if int_digits > max_digits:
                return True
        return False

    def _theta_is_big_for_t(t):
        return _needs_single_col(w_hist[t], max_digits=5)

    force_matrix_for_dense = False
    if d <= threshold_dense:
        for t in range(steps_n):
            if _theta_is_big_for_t(t):
                force_matrix_for_dense = True
                break

    if d <= threshold_dense and not force_matrix_for_dense:

        def model_header_latex():
            return (
                rf"$$\hat{{p}}(y=1\mid \mathbf{{x}})=\sigma(z),\qquad "
                rf"z=\sum_{{j=1}}^{{{d}}}\theta_jx_j+\theta_0$$"
            )

        def full_scalar_model_multiline_latex(t):
            w = w_hist[t]
            b = float(b_hist[t])

            terms = [rf"({w[i]:.{dec}f})x_{{{i+1}}}" for i in range(d)]
            chunks = [terms[i:i + terms_per_line] for i in range(0, len(terms), terms_per_line)]

            lines = []
            lines.append(r"z = " + " + ".join(chunks[0]))
            for ch in chunks[1:]:
                lines.append(r"\quad " + " + ".join(ch))

            lines[-1] = lines[-1] + rf" + ({b:.{dec}f})"
            body = r" \\ ".join(lines)
            return (
                r"$$\begin{aligned}"
                + body
                + r"\\[4pt]\hat{p}(y=1\mid \mathbf{x})=\dfrac{1}{1+e^{-z}}"
                + r"\end{aligned}$$"
            )

        def make_annotations(t):
            ann = [
                dict(
                    x=0.68, y=0.93,
                    xref="paper", yref="paper",
                    text=model_header_latex(),
                    showarrow=False,
                    xanchor="center", yanchor="top",
                    font=dict(size=22, color="white"),
                ),
                dict(
                    x=0.68, y=0.78,
                    xref="paper", yref="paper",
                    text=full_scalar_model_multiline_latex(t),
                    showarrow=False,
                    xanchor="center", yanchor="top",
                    font=dict(size=17, color="white"),
                ),
            ]
            if show_loss:
                ann.append(
                    dict(
                        x=0.33, y=0.94,
                        xref="paper", yref="paper",
                        text=f"<b>Log-loss</b><br>{loss_hist[t]:.6f}",
                        showarrow=False,
                        xanchor="left",
                        yanchor="top",
                        font=dict(size=16, color="black"),
                        bgcolor="white",
                        bordercolor="black",
                        borderwidth=1,
                        borderpad=8,
                    )
                )
            return ann

        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.42, 0.58],
            horizontal_spacing=0.06,
            specs=[[{"type": "xy"}, {"type": "xy"}]],
        )

        fig.add_trace(
            go.Scatter(x=[], y=[], mode="lines", name="Log-loss", line=dict(width=3), uid="LOSS_LINE"),
            row=1, col=1
        )

        frames = []
        for t in range(steps_n):
            loss_trace = (
                go.Scatter(x=step_axis[:t + 1], y=loss_hist[:t + 1], mode="lines", line=dict(width=3), uid="LOSS_LINE")
                if show_loss else go.Scatter(x=[], y=[], uid="LOSS_LINE")
            )
            frames.append(
                go.Frame(
                    name=str(t),
                    data=[loss_trace],
                    traces=[0],
                    layout=go.Layout(annotations=make_annotations(t)),
                )
            )
        fig.frames = frames

        fig.update_layout(
            template="plotly_dark",
            height=760,
            font=dict(family="Helvetica"),
            title=dict(
                text=title,
                x=0.5,
                xanchor="center",
                font=dict(color="white", size=24),
            ),
            margin=dict(l=70, r=40, t=110, b=95),
            showlegend=False,
            sliders=[dict(
                active=0,
                currentvalue=dict(prefix="Step: "),
                pad=dict(t=45),
                steps=[dict(
                    method="animate",
                    args=[[str(t)], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0},
                    }],
                    label=str(t),
                ) for t in range(steps_n)],
            )],
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0.07, y=1.1,
                xanchor="left", yanchor="top",
                bgcolor="white",
                bordercolor="black",
                borderwidth=1,
                font=dict(color="black", size=14),
                buttons=[
                    dict(label="Play", method="animate",
                         args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate",
                         args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )],
            annotations=make_annotations(0),
        )

        fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=1)
        if show_loss:
            fig.update_yaxes(title="Log-loss", range=[lmin - lpad, lmax + lpad], row=1, col=1)
        else:
            fig.update_yaxes(title="Log-loss", row=1, col=1)

        fig.update_xaxes(visible=False, row=1, col=2, range=[0, 1])
        fig.update_yaxes(visible=False, row=1, col=2, range=[0, 1])
        return fig

    rows = 15
    x_cols = 5
    capacity_x = rows * x_cols

    force_theta_one_col = False
    for t in range(steps_n):
        if _theta_is_big_for_t(t):
            force_theta_one_col = True
            break

    def theta_cols_for_t(_t):
        return 1 if force_theta_one_col else 5

    def model_formula_latex():
        return r"$$z=\theta_0+\operatorname{vec}(\boldsymbol{\theta})^\top \operatorname{vec}(\mathbf{x}),\qquad \hat{p}(y=1\mid \mathbf{x})=\sigma(z)$$"

    def bias_latex(t):
        return rf"$$\theta_0 = {float(b_hist[t]):.{dec}f}$$"

    def x_dim_latex():
        return rf"$$\mathbf{{x}} \in \mathbb{{R}}^{{{d}\times {x_cols}}}$$"

    def theta_dim_latex(t):
        th_cols = theta_cols_for_t(t)
        return rf"$$\boldsymbol{{\theta}} \in \mathbb{{R}}^{{{d}\times {th_cols}}}$$"

    def x_vector_latex():
        def cell(j):
            return rf"x_{{{j}}}"

        def vdots_row():
            return " & ".join([r"\vdots"] * x_cols)

        lines = []
        if d <= capacity_x:
            items = [cell(j) for j in range(1, d + 1)] + [r"\;"] * (capacity_x - d)
            M = np.array(items, dtype=object).reshape(rows, x_cols)
            for r in range(rows):
                lines.append(" & ".join(M[r, c] for c in range(x_cols)))
        else:
            head_rows = rows // 2
            tail_rows = rows - head_rows - 1

            head_js = list(range(1, head_rows * x_cols + 1))
            H = np.array([cell(j) for j in head_js], dtype=object).reshape(head_rows, x_cols)
            for r in range(head_rows):
                lines.append(" & ".join(H[r, c] for c in range(x_cols)))

            lines.append(vdots_row())

            tail_count = tail_rows * x_cols
            tail_js = list(range(d - tail_count + 1, d + 1))
            T = np.array([cell(j) for j in tail_js], dtype=object).reshape(tail_rows, x_cols)
            for r in range(tail_rows):
                lines.append(" & ".join(T[r, c] for c in range(x_cols)))

        body = r" \\ ".join(lines)
        return rf"$$\mathbf{{x}} = \begin{{bmatrix}} {body} \end{{bmatrix}}$$"

    def w_matrix_latex(t):
        w = np.asarray(w_hist[t], dtype=float).ravel()
        d_local = w.size
        th_cols = theta_cols_for_t(t)

        def fmt(x):
            return rf"{x:+.{dec}f}"

        if th_cols == 1:
            if d_local <= rows:
                lines = [fmt(w[i]) for i in range(d_local)] + [r"\;"] * (rows - d_local)
            else:
                head_rows = rows // 2
                tail_rows = rows - head_rows - 1
                head_vals = w[:head_rows]
                tail_vals = w[-tail_rows:]
                lines = [fmt(v) for v in head_vals]
                lines.append(r"\vdots")
                lines += [fmt(v) for v in tail_vals]

            body = r" \\ ".join(lines)
            return rf"$$\boldsymbol{{\theta}} = \begin{{bmatrix}} {body} \end{{bmatrix}}$$"

        th_capacity = rows * th_cols

        def vdots_row():
            return " & ".join([r"\vdots"] * th_cols)

        lines = []
        if d_local <= th_capacity:
            padded = np.full(th_capacity, np.nan, dtype=float)
            padded[:d_local] = w
            W = padded.reshape(rows, th_cols)

            for r in range(rows):
                row_items = []
                for c in range(th_cols):
                    row_items.append(r"\;" if np.isnan(W[r, c]) else fmt(W[r, c]))
                lines.append(" & ".join(row_items))
        else:
            head_rows = rows // 2
            tail_rows = rows - head_rows - 1

            head_vals = w[: head_rows * th_cols]
            tail_vals = w[-(tail_rows * th_cols):]

            H = head_vals.reshape(head_rows, th_cols)
            for r in range(head_rows):
                lines.append(" & ".join(fmt(H[r, c]) for c in range(th_cols)))

            lines.append(vdots_row())

            T = tail_vals.reshape(tail_rows, th_cols)
            for r in range(tail_rows):
                lines.append(" & ".join(fmt(T[r, c]) for c in range(th_cols)))

        body = r" \\ ".join(lines)
        return rf"$$\boldsymbol{{\theta}} = \begin{{bmatrix}} {body} \end{{bmatrix}}$$"

    def scalar_model_compact_latex(t):
        w = np.asarray(w_hist[t], dtype=float).ravel()
        b = float(b_hist[t])
        last = d
        th_cols = theta_cols_for_t(t)

        if th_cols == 1:
            z_part = (
                r"z = "
                + rf"({w[0]:.{dec}f})x_1 "
                + rf"+ \cdots + ({w[last-1]:.{dec}f})x_{{{last}}} "
                + rf"+ ({b:.{dec}f})"
            )
        else:
            z_part = (
                r"z = "
                + rf"({w[0]:.{dec}f})x_1 "
                + rf"+ ({w[1]:.{dec}f})x_2 "
                + rf"+ ({w[2]:.{dec}f})x_3 "
                + rf"+ ({w[3]:.{dec}f})x_4 "
                + rf"+ \cdots + ({w[last-1]:.{dec}f})x_{{{last}}} "
                + rf"+ ({b:.{dec}f})"
            )

        return r"$$" + z_part + r",\qquad \hat{p}(y=1\mid \mathbf{x})=\dfrac{1}{1+e^{-z}} $$"

    def make_annotations(t):
        ann = [
            dict(
                x=0.68, y=0.995,
                xref="paper", yref="paper",
                text=model_formula_latex(),
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=22, color="white"),
            ),
            dict(
                x=0.68, y=0.938,
                xref="paper", yref="paper",
                text=bias_latex(t),
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=18, color="white"),
            ),
            dict(
                x=0.55, y=0.83,
                xref="paper", yref="paper",
                text=x_dim_latex(),
                showarrow=False,
                xanchor="center", yanchor="bottom",
                font=dict(size=14, color="white"),
            ),
            dict(
                x=0.83, y=0.83,
                xref="paper", yref="paper",
                text=theta_dim_latex(t),
                showarrow=False,
                xanchor="center", yanchor="bottom",
                font=dict(size=14, color="white"),
            ),
            dict(
                x=0.52, y=0.48,
                xref="paper", yref="paper",
                text=x_vector_latex(),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=15, color="white"),
            ),
            dict(
                x=0.80, y=0.48,
                xref="paper", yref="paper",
                text=w_matrix_latex(t),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=15, color="white"),
            ),
            dict(
                x=0.71, y=0.03,
                xref="paper", yref="paper",
                text=scalar_model_compact_latex(t),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=16, color="white"),
            ),
        ]

        if show_loss:
            th_cols = theta_cols_for_t(t)
            y_loss = 0.98 if th_cols == 1 else 0.86
            ann.append(
                dict(
                    x=0.25, y=y_loss,
                    xref="paper", yref="paper",
                    text=f"<b>Log-loss</b><br>{loss_hist[t]:.6f}",
                    showarrow=False,
                    xanchor="left",
                    yanchor="top",
                    font=dict(size=16, color="black"),
                    bgcolor="white",
                    bordercolor="black",
                    borderwidth=1,
                    borderpad=8,
                )
            )

        return ann

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.42, 0.58],
        horizontal_spacing=0.06,
        specs=[[{"type": "xy"}, {"type": "xy"}]],
    )

    fig.add_trace(
        go.Scatter(x=[], y=[], mode="lines", name="Log-loss", line=dict(width=3), uid="LOSS_LINE"),
        row=1, col=1
    )

    frames = []
    for t in range(steps_n):
        loss_trace = (
            go.Scatter(x=step_axis[:t + 1], y=loss_hist[:t + 1], mode="lines", line=dict(width=3), uid="LOSS_LINE")
            if show_loss else go.Scatter(x=[], y=[], uid="LOSS_LINE")
        )
        frames.append(
            go.Frame(
                name=str(t),
                data=[loss_trace],
                traces=[0],
                layout=go.Layout(annotations=make_annotations(t)),
            )
        )
    fig.frames = frames

    fig.update_layout(
        template="plotly_dark",
        font=dict(family="Helvetica"),
        height=760,
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(color="white", size=24),
        ),
        margin=dict(l=70, r=40, t=110, b=95),
        showlegend=False,
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=45),
            steps=[dict(
                method="animate",
                args=[[str(t)], {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0},
                }],
                label=str(t),
            ) for t in range(steps_n)],
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.07, y=1.1,
            xanchor="left", yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(color="black", size=14),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
        annotations=make_annotations(0),
    )

    fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=1)
    if show_loss:
        fig.update_yaxes(title="Log-loss", range=[lmin - lpad, lmax + lpad], row=1, col=1)
    else:
        fig.update_yaxes(title="Log-loss", row=1, col=1)

    fig.update_xaxes(visible=False, row=1, col=2, range=[0, 1])
    fig.update_yaxes(visible=False, row=1, col=2, range=[0, 1])
    return fig


# ============================================================
# Multiclass logistic (d = 1)
# ============================================================
def build_multiclass_1d_logistic_figure(
    x1, y,
    w_hist, b_hist,
    *,
    p_curves_hist=None,
    x1_grid=None,
    loss_hist=None,
    show_loss=True,
    history_kind="iterative",
    title=None,
    strict_loss=False,
    dec=3,
    example_class=0,
    max_theta_cols=8,
):
    if show_loss and history_kind != "iterative":
        if strict_loss:
            raise ValueError("show_loss=True is only allowed for iterative histories.")
        show_loss = False
        loss_hist = None

    x1 = np.asarray(x1).ravel()
    y = np.asarray(y).ravel()

    w_hist = np.asarray(w_hist, dtype=float)   # (T, 1, K)
    b_hist = np.asarray(b_hist, dtype=float)   # (T, K)

    if w_hist.ndim != 3 or w_hist.shape[1] != 1:
        raise ValueError("For multiclass 1D, w_hist must have shape (steps, 1, K).")
    if b_hist.ndim != 2:
        raise ValueError("For multiclass 1D, b_hist must have shape (steps, K).")
    if w_hist.shape[0] != b_hist.shape[0] or w_hist.shape[2] != b_hist.shape[1]:
        raise ValueError("w_hist and b_hist shapes are inconsistent.")

    steps_n = int(w_hist.shape[0])
    K = int(w_hist.shape[2])

    if title is None:
        title = f"Multiclass Logistic Regression (K={K}, d=1)"

    if p_curves_hist is not None:
        p_curves_hist = np.asarray(p_curves_hist, dtype=float)
        if x1_grid is None:
            raise ValueError("If p_curves_hist is provided, x1_grid must be provided.")
        x1_grid = np.asarray(x1_grid, dtype=float).ravel()

        if p_curves_hist.ndim != 3:
            raise ValueError("p_curves_hist must have shape (steps, grid_points, K).")
        if p_curves_hist.shape != (steps_n, x1_grid.size, K):
            raise ValueError("p_curves_hist shape mismatch with steps/x1_grid/K.")

        def p_curves(t):
            return p_curves_hist[t]

    else:
        x_min, x_max = float(x1.min()), float(x1.max())
        x1_grid = np.linspace(x_min, x_max, 350)

        def p_curves(t):
            Zg = x1_grid.reshape(-1, 1) @ w_hist[t] + b_hist[t].reshape(1, -1)
            return _softmax(Zg)

    if show_loss:
        if loss_hist is None:
            raise ValueError("show_loss=True requires loss_hist.")
        loss_hist = np.asarray(loss_hist, dtype=float).ravel()
        if loss_hist.size != steps_n:
            raise ValueError("loss_hist must match steps.")

    ep = np.arange(steps_n)

    def model_formula_latex():
        return r"$$\mathbf{z}=\Theta^\top\mathbf{x},\qquad \hat{\mathbf{p}}=\mathrm{softmax}(\mathbf{z})$$"

    def x_definition_latex():
        return r"$$\mathbf{x}=\begin{bmatrix}x\\1\end{bmatrix}\in\mathbb{R}^{2}$$"

    def softmax_def_latex():
        return rf"$$\mathrm{{softmax}}(\mathbf{{z}})_k=\dfrac{{e^{{z_k}}}}{{\sum_{{j=1}}^{{{K}}}e^{{z_j}}}},\;\;k=1,\dots,{K}$$"

    def theta_definition_latex():
        return rf"$$\Theta\in\mathbb{{R}}^{{2\times {K}}},\quad z_k(x)=\theta_{{1,k}}x+\theta_{{0,k}}$$"

    def theta_matrix_latex_math_style(t, max_elems=max_theta_cols, dec=dec):
        Theta = np.vstack([w_hist[t, 0], b_hist[t]])  # (2,K)
        K_local = Theta.shape[1]

        def fmt(v):
            return rf"{v:+.{dec}f}"

        if K_local <= max_elems:
            row1 = " & ".join(fmt(Theta[0, j]) for j in range(K_local))
            row2 = " & ".join(fmt(Theta[1, j]) for j in range(K_local))
            cols_spec = "c" * K_local
            return (
                r"$$"
                r"\Theta=\left[\begin{array}{" + cols_spec + r"}"
                + row1 + r"\\"
                + row2 +
                r"\end{array}\right]"
                r"$$"
            )

        head = (max_elems - 1) // 2
        tail = (max_elems - 1) - head
        head_idx = list(range(head))
        tail_idx = list(range(K_local - tail, K_local))

        row1_items = [fmt(Theta[0, j]) for j in head_idx] + [r"\cdots"] + [fmt(Theta[0, j]) for j in tail_idx]
        row2_items = [fmt(Theta[1, j]) for j in head_idx] + [r"\cdots"] + [fmt(Theta[1, j]) for j in tail_idx]

        row1 = " & ".join(row1_items)
        row2 = " & ".join(row2_items)
        cols_spec = "c" * max_elems

        return (
            r"$$"
            r"\Theta=\left[\begin{array}{" + cols_spec + r"}"
            + row1 + r"\\"
            + row2 +
            r"\end{array}\right]"
            r"$$"
        )

    def z_numeric_expr_univar(Theta, class_idx, dec=dec):
        def num(v):
            return f"{v:+.{dec}f}"
        theta_1k = num(Theta[0, class_idx])
        theta_0k = num(Theta[1, class_idx])
        return rf"\left({theta_1k}\right)x + \left({theta_0k}\right)"

    def denom_three_terms_tex(Theta, K_local, dec=dec):
        z1 = z_numeric_expr_univar(Theta, 0, dec=dec)
        if K_local == 1:
            return rf"e^{{{z1}}}"

        z2 = z_numeric_expr_univar(Theta, 1, dec=dec)
        if K_local == 2:
            return rf"e^{{{z1}}} + e^{{{z2}}}"

        zK = z_numeric_expr_univar(Theta, K_local - 1, dec=dec)
        return rf"e^{{{z1}}} + e^{{{z2}}} + \cdots + e^{{{zK}}}"

    def final_prob_example_latex(t, class_k=example_class, dec=dec):
        Theta = np.vstack([w_hist[t, 0], b_hist[t]])
        K_local = Theta.shape[1]

        k = int(class_k)
        k = max(0, min(k, K_local - 1))

        z_k = z_numeric_expr_univar(Theta, k, dec=dec)
        num_tex = rf"e^{{{z_k}}}"
        denom_tex = denom_three_terms_tex(Theta, K_local, dec=dec)

        return (
            r"$$"
            r"\begin{aligned}"
            + rf"\hat{{p}}(y=1\mid x) &= \frac{{e^{{z_1(x)}}}}{{\sum_{{j=1}}^{{{K_local}}} e^{{z_j(x)}}}} \\[6pt]"
            + rf"&= \frac{{{num_tex}}}{{{denom_tex}}}"
            r"\end{aligned}"
            r"$$"
        )

    def vertical_dots_latex():
        return r"$$\vdots$$"

    def last_class_tail_latex(t, dec=dec):
        Theta = np.vstack([w_hist[t, 0], b_hist[t]])
        K_local = Theta.shape[1]
        last_idx = K_local - 1

        z_last = z_numeric_expr_univar(Theta, last_idx, dec=dec)
        num_tex = rf"e^{{{z_last}}}"
        denom_tex = denom_three_terms_tex(Theta, K_local, dec=dec)

        return (
            r"$$"
            r"\begin{aligned}"
            + rf"\hat{{p}}(y={K_local}\mid x) &= \frac{{{num_tex}}}{{{denom_tex}}}"
            r"\end{aligned}"
            r"$$"
        )

    if show_loss:
        fig = make_subplots(
            rows=1, cols=3,
            column_widths=[0.22, 0.18, 0.28],
            horizontal_spacing=0.06,
            specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]],
        )

        Pg0 = p_curves(0)
        for k in range(K):
            fig.add_trace(
                go.Scatter(
                    x=x1_grid, y=Pg0[:, k],
                    mode="lines",
                    name=f"p(class {k})",
                    line=dict(width=4),
                    legendgroup="curves",
                    showlegend=True,
                ),
                row=1, col=1
            )

        fig.add_trace(
            go.Scatter(
                x=[0], y=[loss_hist[0]],
                mode="lines",
                name="Cross-entropy",
                line=dict(width=3),
                legendgroup="loss",
                showlegend=True,
            ),
            row=1, col=2
        )

        X_TEXT = 0.78
        X_VDOTS = 0.82

        def make_annotations(t):
            return [
                dict(x=X_TEXT, y=0.955, xref="paper", yref="paper", text=model_formula_latex(),
                     showarrow=False, xanchor="center", yanchor="top", font=dict(size=20, color="white")),
                dict(x=X_TEXT, y=0.895, xref="paper", yref="paper", text=x_definition_latex(),
                     showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
                dict(x=X_TEXT, y=0.805, xref="paper", yref="paper", text=softmax_def_latex(),
                     showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
                dict(x=X_TEXT, y=0.700, xref="paper", yref="paper", text=theta_definition_latex(),
                     showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
                dict(x=X_TEXT, y=0.575, xref="paper", yref="paper", text=theta_matrix_latex_math_style(t),
                     showarrow=False, xanchor="center", yanchor="middle", font=dict(size=20, color="white")),
                dict(x=X_TEXT, y=0.340, xref="paper", yref="paper", text=final_prob_example_latex(t),
                     showarrow=False, xanchor="center", yanchor="middle", font=dict(size=16, color="white")),
                dict(x=X_VDOTS, y=0.200, xref="paper", yref="paper", text=vertical_dots_latex(),
                     showarrow=False, xanchor="center", yanchor="middle", font=dict(size=22, color="white")),
                dict(x=X_TEXT, y=0.095, xref="paper", yref="paper", text=last_class_tail_latex(t),
                     showarrow=False, xanchor="center", yanchor="middle", font=dict(size=16, color="white")),
            ]

        frames = []
        for t in range(steps_n):
            Pg = p_curves(t)
            curve_updates = [go.Scatter(x=x1_grid, y=Pg[:, k]) for k in range(K)]
            loss_update = go.Scatter(x=ep[:t + 1], y=loss_hist[:t + 1])

            frames.append(go.Frame(
                name=str(t),
                data=curve_updates + [loss_update],
                traces=list(range(0, K + 1)),
                layout=go.Layout(annotations=make_annotations(t))
            ))
        fig.frames = frames

        loss_min, loss_max = float(loss_hist.min()), float(loss_hist.max())
        loss_pad = 0.08 * ((loss_max - loss_min) + 1e-9)

        fig.update_layout(
            template="plotly_dark",
            height=840,
            title=dict(
                text=title,
                x=0.5,
                xanchor="center",
                font=dict(color="white", size=24),
            ),
            font=dict(family="Helvetica"),
            margin=dict(l=70, r=40, t=110, b=95),
            legend=dict(
                orientation="v",
                x=0.28, y=0.9,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(230,230,230,0.85)",
                bordercolor="rgba(0,0,0,0.35)",
                borderwidth=1,
                font=dict(size=12, color="black"),
            ),
            legend2=dict(
                orientation="v",
                x=0.58, y=0.9,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(230,230,230,0.85)",
                bordercolor="rgba(0,0,0,0.35)",
                borderwidth=1,
                font=dict(size=12, color="black"),
            ),
            sliders=[dict(
                active=0,
                currentvalue=dict(prefix="Step: "),
                pad=dict(t=45),
                steps=[dict(
                    method="animate",
                    args=[[str(t)], {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0}
                    }],
                    label=str(t)
                ) for t in range(steps_n)],
            )],
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0.07, y=0.98,
                xanchor="left", yanchor="top",
                bgcolor="white",
                bordercolor="black",
                borderwidth=1,
                font=dict(color="black", size=14),
                buttons=[
                    dict(label="Play", method="animate",
                         args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                    dict(label="Pause", method="animate",
                         args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
                ],
            )],
            annotations=make_annotations(0),
        )

        fig.data[K].update(legend="legend2")
        fig.update_xaxes(title=r"$x$", row=1, col=1)
        fig.update_yaxes(title="Probability", range=[-0.02, 1.02], row=1, col=1)
        fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=2)
        fig.update_yaxes(title="Cross-entropy", range=[loss_min - loss_pad, loss_max + loss_pad], row=1, col=2)
        fig.update_xaxes(visible=False, row=1, col=3, range=[0, 1])
        fig.update_yaxes(visible=False, row=1, col=3, range=[0, 1])
        return fig

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.58, 0.42],
        horizontal_spacing=0.06,
        specs=[[{"type": "xy"}, {"type": "xy"}]],
    )

    Pg0 = p_curves(0)
    for k in range(K):
        fig.add_trace(
            go.Scatter(
                x=x1_grid, y=Pg0[:, k],
                mode="lines",
                name=f"p(class {k})",
                line=dict(width=4),
            ),
            row=1, col=1
        )

    X_TEXT = 0.78
    X_VDOTS = 0.82

    def make_annotations_no_loss(t):
        return [
            dict(x=X_TEXT, y=0.955, xref="paper", yref="paper", text=model_formula_latex(),
                 showarrow=False, xanchor="center", yanchor="top", font=dict(size=20, color="white")),
            dict(x=X_TEXT, y=0.885, xref="paper", yref="paper", text=x_definition_latex(),
                 showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
            dict(x=X_TEXT, y=0.785, xref="paper", yref="paper", text=softmax_def_latex(),
                 showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
            dict(x=X_TEXT, y=0.665, xref="paper", yref="paper", text=theta_definition_latex(),
                 showarrow=False, xanchor="center", yanchor="top", font=dict(size=18, color="white")),
            dict(x=X_TEXT, y=0.53, xref="paper", yref="paper", text=theta_matrix_latex_math_style(t),
                 showarrow=False, xanchor="center", yanchor="middle", font=dict(size=20, color="white")),
            dict(x=X_TEXT, y=0.29, xref="paper", yref="paper", text=final_prob_example_latex(t),
                 showarrow=False, xanchor="center", yanchor="middle", font=dict(size=16, color="white")),
            dict(x=X_VDOTS, y=0.17, xref="paper", yref="paper", text=vertical_dots_latex(),
                 showarrow=False, xanchor="center", yanchor="middle", font=dict(size=22, color="white")),
            dict(x=X_TEXT, y=0.08, xref="paper", yref="paper", text=last_class_tail_latex(t),
                 showarrow=False, xanchor="center", yanchor="middle", font=dict(size=16, color="white")),
        ]

    frames = []
    for t in range(steps_n):
        Pg = p_curves(t)
        frame_data = [go.Scatter(x=x1_grid, y=Pg[:, k]) for k in range(K)]
        frames.append(go.Frame(
            name=str(t),
            data=frame_data,
            traces=list(range(0, K)),
            layout=go.Layout(annotations=make_annotations_no_loss(t))
        ))
    fig.frames = frames

    fig.update_layout(
        template="plotly_dark",
        height=840,
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(color="white", size=24),
        ),
        margin=dict(l=70, r=40, t=110, b=95),
        legend=dict(
            orientation="v",
            x=0.55, y=0.9,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(230,230,230,0.85)",
            bordercolor="rgba(0,0,0,0.35)",
            borderwidth=1,
            font=dict(size=12, color="black"),
        ),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=45),
            steps=[dict(
                method="animate",
                args=[[str(t)], {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0}
                }],
                label=str(t)
            ) for t in range(steps_n)],
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.07, y=0.98,
            xanchor="left", yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(color="black", size=14),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
        annotations=make_annotations_no_loss(0),
    )

    fig.update_xaxes(title=r"$x$", row=1, col=1)
    fig.update_yaxes(title="Probability", range=[-0.02, 1.02], row=1, col=1)
    fig.update_xaxes(visible=False, row=1, col=2, range=[0, 1])
    fig.update_yaxes(visible=False, row=1, col=2, range=[0, 1])
    return fig


# ============================================================
# Multiclass logistic multivariable (d > 1)
# ============================================================
def build_multiclass_multivar_logistic_figure(
    X, y,
    w_hist, b_hist,
    *,
    loss_hist=None,
    show_loss=True,
    history_kind="iterative",
    title=None,
    strict_loss=False,
    dec=3,
    example_class=0,
    max_features_in_z=3,
):
    if show_loss and history_kind != "iterative":
        if strict_loss:
            raise ValueError("show_loss=True is only allowed for iterative histories.")
        show_loss = False
        loss_hist = None

    X = np.asarray(X)
    y = np.asarray(y).ravel()
    w_hist = np.asarray(w_hist, dtype=float)  # (T, d, K)
    b_hist = np.asarray(b_hist, dtype=float)  # (T, K)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if w_hist.ndim != 3:
        raise ValueError("For multiclass multivar, w_hist must have shape (steps, d, K).")
    if b_hist.ndim != 2:
        raise ValueError("For multiclass multivar, b_hist must have shape (steps, K).")
    if w_hist.shape[0] != b_hist.shape[0] or w_hist.shape[2] != b_hist.shape[1]:
        raise ValueError("w_hist and b_hist shapes are inconsistent.")

    steps_n = int(w_hist.shape[0])
    d = int(w_hist.shape[1])
    K = int(w_hist.shape[2])

    if X.shape[1] != d:
        raise ValueError(f"X has d={X.shape[1]} but w_hist has d={d}.")

    if title is None:
        title = f"Multiclass Logistic Regression (K={K}, d={d})"

    if show_loss:
        if loss_hist is None:
            raise ValueError("show_loss=True requires loss_hist.")
        loss_hist = np.asarray(loss_hist, dtype=float).ravel()
        if loss_hist.size != steps_n:
            raise ValueError("loss_hist must match steps.")

    ep = np.arange(steps_n)

    if show_loss:
        loss_min, loss_max = float(loss_hist.min()), float(loss_hist.max())
        loss_pad = 0.08 * ((loss_max - loss_min) + 1e-9)

    def model_formula_latex():
        return r"$$\mathbf{z}=\Theta^\top\mathbf{x},\qquad \hat{\mathbf{p}}=\mathrm{softmax}(\mathbf{z})$$"

    def softmax_def_latex():
        return rf"$$\mathrm{{softmax}}(\mathbf{{z}})_k=\dfrac{{e^{{z_k}}}}{{\sum_{{j=1}}^{{{K}}}e^{{z_j}}}},\;\;k=1,\dots,{K}$$"

    def Theta_definition_latex():
        return (
            rf"$$\Theta\in\mathbb{{R}}^{{({d}+1)\times {K}}},\quad "
            rf"z_k(\mathbf{{x}})=\sum_{{j=1}}^{{{d+1}}}\theta_{{j,k}}x_j$$"
        )

    def x_vector_latex_capped(d_local, max_rows=7, max_cols=4):
        entries = [rf"x_{{{j}}}" for j in range(1, d_local + 1)] + [r"1"]
        D = d_local + 1
        capacity = max_rows * max_cols

        def vdots_row():
            return " & ".join([r"\vdots"] * max_cols)

        lines = []

        if D <= capacity:
            padded = entries + [r"\;"] * (capacity - D)
            M = np.array(padded, dtype=object).reshape(max_rows, max_cols)
            for r in range(max_rows):
                lines.append(" & ".join(M[r, c] for c in range(max_cols)))
        else:
            head_rows = max(2, max_rows // 2 - 1)
            tail_rows = max_rows - head_rows - 1

            head_count = head_rows * max_cols
            tail_count = tail_rows * max_cols

            head_items = entries[:head_count]
            tail_items = entries[-tail_count:]

            H = np.array(head_items, dtype=object).reshape(head_rows, max_cols)
            for r in range(head_rows):
                lines.append(" & ".join(H[r, c] for c in range(max_cols)))

            lines.append(vdots_row())

            T = np.array(tail_items, dtype=object).reshape(tail_rows, max_cols)
            for r in range(tail_rows):
                lines.append(" & ".join(T[r, c] for c in range(max_cols)))

        body = r" \\ ".join(lines)
        return rf"$$\mathbf{{x}}=\begin{{bmatrix}} {body} \end{{bmatrix}}$$"

    def Theta_matrix_latex_capped(t, max_rows=7, max_cols=6, dec=dec):
        Theta = np.vstack([w_hist[t], b_hist[t].reshape(1, -1)])  # (d+1, K)
        R, C = Theta.shape

        def fmt(v):
            return rf"{v:+.{dec}f}"

        if R <= max_rows:
            row_slots = list(range(R))
        else:
            head_r = (max_rows - 1) // 2
            tail_r = max_rows - head_r - 1
            row_slots = list(range(head_r)) + [None] + list(range(R - tail_r, R))

        if C <= max_cols:
            col_slots = list(range(C))
        else:
            head_c = (max_cols - 1) // 2
            tail_c = max_cols - head_c - 1
            col_slots = list(range(head_c)) + [None] + list(range(C - tail_c, C))

        lines = []
        for r in row_slots:
            items = []
            for c in col_slots:
                if r is None and c is None:
                    items.append(r"\ddots")
                elif r is None:
                    items.append(r"\vdots")
                elif c is None:
                    items.append(r"\cdots")
                else:
                    items.append(fmt(Theta[r, c]))
            lines.append(" & ".join(items))

        body = r" \\ ".join(lines)
        cols_spec = "c" * len(col_slots)

        return (
            r"$$"
            r"\Theta=\left[\begin{array}{" + cols_spec + r"}"
            + body +
            r"\end{array}\right]"
            r"$$"
        )

    def z_numeric_expr(Theta, class_idx, d_local, max_feat=max_features_in_z, dec=dec):
        def num(v):
            return f"{v:+.{dec}f}"

        feat_count = min(d_local, max_feat)
        terms = [rf"\left({num(Theta[j, class_idx])}\right)x_{{{j+1}}}" for j in range(feat_count)]
        if d_local > max_feat:
            terms.append(r"\cdots")
        terms.append(rf"\left({num(Theta[d_local, class_idx])}\right)")
        return r" + ".join(terms)

    def final_prob_example_latex(t, class_k=example_class, max_feat=max_features_in_z, dec=dec):
        Theta = np.vstack([w_hist[t], b_hist[t].reshape(1, -1)])
        D, K_local = Theta.shape
        d_local = D - 1

        k = int(class_k)
        k = max(0, min(k, K_local - 1))

        z_k = z_numeric_expr(Theta, k, d_local, max_feat=max_feat, dec=dec)
        num_tex = rf"e^{{{z_k}}}"

        z_first = z_numeric_expr(Theta, 0, d_local, max_feat=max_feat, dec=dec)
        z_last = z_numeric_expr(Theta, K_local - 1, d_local, max_feat=max_feat, dec=dec)

        if K_local == 1:
            denom_tex = rf"e^{{{z_first}}}"
        else:
            denom_tex = rf"e^{{{z_first}}} + \cdots + e^{{{z_last}}}"

        return (
            r"$$"
            r"\begin{aligned}"
            + rf"\hat{{p}}(y=1\mid \mathbf{{x}}) &= \frac{{e^{{z_1(\mathbf{{x}})}}}}{{\sum_{{j=1}}^{{{K_local}}} e^{{z_j(\mathbf{{x}})}}}} \\[6pt]"
            + rf"&= \frac{{{num_tex}}}{{{denom_tex}}}"
            r"\end{aligned}"
            r"$$"
        )

    def vertical_dots_latex():
        return r"$$\vdots$$"

    def last_class_tail_latex(t, max_feat=max_features_in_z, dec=dec):
        Theta = np.vstack([w_hist[t], b_hist[t].reshape(1, -1)])
        D, K_local = Theta.shape
        d_local = D - 1

        last_idx = K_local - 1
        z_last = z_numeric_expr(Theta, last_idx, d_local, max_feat=max_feat, dec=dec)
        num_tex = rf"e^{{{z_last}}}"

        z_first = z_numeric_expr(Theta, 0, d_local, max_feat=max_feat, dec=dec)
        if K_local == 1:
            denom_tex = rf"e^{{{z_first}}}"
        else:
            denom_tex = rf"e^{{{z_first}}} + \cdots + e^{{{z_last}}}"

        return (
            r"$$"
            r"\begin{aligned}"
            + rf"\hat{{p}}(y={K_local}\mid \mathbf{{x}}) &= \frac{{{num_tex}}}{{{denom_tex}}}"
            r"\end{aligned}"
            r"$$"
        )

    fig = make_subplots(
        rows=1, cols=2,
        column_widths=[0.55, 0.45],
        horizontal_spacing=0.06,
        specs=[[{"type": "xy"}, {"type": "xy"}]],
    )

    if show_loss:
        fig.add_trace(
            go.Scatter(x=[], y=[], mode="lines", name="Cross-entropy", line=dict(width=3)),
            row=1, col=1
        )
    else:
        fig.add_trace(
            go.Scatter(x=[], y=[], mode="lines", name="Cross-entropy", line=dict(width=3)),
            row=1, col=1
        )

    def make_annotations(t):
        ann = [
            dict(
                x=0.74, y=0.965,
                xref="paper", yref="paper",
                text=model_formula_latex(),
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=20, color="white"),
            ),
            dict(
                x=0.74, y=0.885,
                xref="paper", yref="paper",
                text=softmax_def_latex(),
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=18, color="white"),
            ),
            dict(
                x=0.74, y=0.775,
                xref="paper", yref="paper",
                text=Theta_definition_latex(),
                showarrow=False,
                xanchor="center", yanchor="top",
                font=dict(size=18, color="white"),
            ),
            dict(
                x=0.64, y=0.49,
                xref="paper", yref="paper",
                text=x_vector_latex_capped(d, max_rows=7, max_cols=4),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=16, color="white"),
            ),
            dict(
                x=0.84, y=0.49,
                xref="paper", yref="paper",
                text=Theta_matrix_latex_capped(t, max_rows=7, max_cols=6, dec=dec),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=16, color="white"),
            ),
            dict(
                x=0.76, y=0.22,
                xref="paper", yref="paper",
                text=final_prob_example_latex(t, class_k=example_class, max_feat=max_features_in_z, dec=dec),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=16, color="white"),
            ),
            dict(
                x=0.81, y=0.095,
                xref="paper", yref="paper",
                text=vertical_dots_latex(),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=22, color="white"),
            ),
            dict(
                x=0.76, y=0.013,
                xref="paper", yref="paper",
                text=last_class_tail_latex(t, max_feat=max_features_in_z, dec=dec),
                showarrow=False,
                xanchor="center", yanchor="middle",
                font=dict(size=16, color="white"),
            ),
        ]

        if show_loss:
            ann.append(
                dict(
                    x=0.535, y=0.94,
                    xref="paper", yref="paper",
                    text=f"<b>Cross-entropy</b><br>{loss_hist[t]:.6f}",
                    showarrow=False,
                    xanchor="right",
                    yanchor="top",
                    font=dict(size=16, color="black"),
                    bgcolor="white",
                    bordercolor="black",
                    borderwidth=1,
                    borderpad=8,
                )
            )
        return ann

    frames = []
    for t in range(steps_n):
        trace = (
            go.Scatter(x=ep[: t + 1], y=loss_hist[: t + 1])
            if show_loss else go.Scatter(x=[], y=[])
        )
        frames.append(
            go.Frame(
                name=str(t),
                data=[trace],
                traces=[0],
                layout=go.Layout(annotations=make_annotations(t)),
            )
        )
    fig.frames = frames

    fig.update_layout(
        template="plotly_dark",
        height=760,
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(color="white", size=24),
        ),
        margin=dict(l=70, r=40, t=110, b=95),
        showlegend=False,
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Step: "),
            pad=dict(t=45),
            steps=[dict(
                method="animate",
                args=[[str(t)], {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": True},
                    "transition": {"duration": 0},
                }],
                label=str(t),
            ) for t in range(steps_n)],
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.07, y=0.98,
            xanchor="left", yanchor="top",
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(color="black", size=14),
            buttons=[
                dict(label="Play", method="animate",
                     args=[None, {"frame": {"duration": 80, "redraw": True}, "transition": {"duration": 0}}]),
                dict(label="Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]),
            ],
        )],
        annotations=make_annotations(0),
    )

    if show_loss:
        fig.update_xaxes(title="Step", range=[0, steps_n - 1], row=1, col=1)
        fig.update_yaxes(title="Cross-entropy", range=[loss_min - loss_pad, loss_max + loss_pad], row=1, col=1)
    else:
        fig.update_xaxes(title="Step", row=1, col=1)
        fig.update_yaxes(title="Cross-entropy", row=1, col=1)

    fig.update_xaxes(visible=False, row=1, col=2, range=[0, 1])
    fig.update_yaxes(visible=False, row=1, col=2, range=[0, 1])
    return fig


# ============================================================
# Routing
# ============================================================
def build_logistic_figure(
    X, y,
    w_hist=None, b_hist=None,
    *,
    history=None,
    p_line_hist=None, x1_grid=None,          # binary d==1
    p_plane_hist=None, X1g=None, X2g=None,   # binary d==2
    p_curves_hist=None,                      # multiclass d==1
    loss_hist=None,
    classes=None,
    show_loss=False,
    history_kind="iterative",
    title=None,
    strict_loss=False,
    dec=4,
):
    X = np.asarray(X)
    y = np.asarray(y).ravel()
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    d = int(X.shape[1])

    if history is not None:
        if not isinstance(history, dict):
            raise ValueError("history must be a dict returned by fit_history_logistic().")

        history_kind = history.get("history_kind", history_kind)
        classes = _first_not_none(history.get("classes", None), classes)
        loss_hist = _first_not_none(
            history.get("loss_hist", None),
            history.get("losses", None),
            history.get("loss", None),
            loss_hist,
        )

        grid = history.get("grid", {}) or {}

        w_hist = _first_not_none(history.get("w_hist", None), w_hist)
        b_hist = _first_not_none(history.get("b_hist", None), b_hist)

        p_line_hist = _first_not_none(history.get("p_line_hist", None), p_line_hist)
        p_plane_hist = _first_not_none(history.get("p_plane_hist", None), p_plane_hist)
        p_curves_hist = _first_not_none(history.get("p_curves_hist", None), p_curves_hist)

        x1_grid = _first_not_none(grid.get("x1_grid", None), x1_grid)
        X1g = _first_not_none(grid.get("X1g", None), X1g)
        X2g = _first_not_none(grid.get("X2g", None), X2g)

    classes = np.unique(y) if classes is None else np.asarray(classes)
    K = len(classes)
    is_multiclass = K > 2

    if not is_multiclass:
        if d == 1:
            x1 = X[:, 0]
            if title is None:
                title = "Binary Logistic Regression (1 variable)"
            return build_binary_simple_logistic_figure(
                x1, y,
                w_hist=w_hist, b_hist=b_hist,
                p_line_hist=p_line_hist,
                x1_grid=x1_grid,
                loss_hist=loss_hist,
                show_loss=show_loss,
                history_kind=history_kind,
                title=title,
                strict_loss=strict_loss,
                dec=dec,
            )

        if d == 2:
            x1 = X[:, 0]
            x2 = X[:, 1]
            if title is None:
                title = "Binary Logistic Regression (2 variables)"
            return build_binary_plane_logistic_figure(
                x1, x2, y,
                w_hist=w_hist, b_hist=b_hist,
                p_plane_hist=p_plane_hist,
                X1g=X1g, X2g=X2g,
                loss_hist=loss_hist,
                show_loss=show_loss,
                history_kind=history_kind,
                title=title,
                strict_loss=strict_loss,
                dec=dec,
            )

        if title is None:
            title = f"Binary Logistic Regression ({d} variables)"
        return build_binary_multivar_logistic_figure(
            X, y,
            w_hist, b_hist,
            loss_hist=loss_hist,
            show_loss=show_loss,
            history_kind=history_kind,
            title=title,
            strict_loss=strict_loss,
            dec=dec,
        )

    # multiclass
    if d == 1:
        if title is None:
            title = f"Multiclass Logistic Regression (K={K}, d=1)"
        return build_multiclass_1d_logistic_figure(
            X[:, 0], y,
            w_hist, b_hist,
            p_curves_hist=p_curves_hist,
            x1_grid=x1_grid,
            loss_hist=loss_hist,
            show_loss=show_loss,
            history_kind=history_kind,
            title=title,
            strict_loss=strict_loss,
            dec=min(dec, 4),
        )

    if title is None:
        title = f"Multiclass Logistic Regression (K={K}, d={d})"
    return build_multiclass_multivar_logistic_figure(
        X, y,
        w_hist, b_hist,
        loss_hist=loss_hist,
        show_loss=show_loss,
        history_kind=history_kind,
        title=title,
        strict_loss=strict_loss,
        dec=min(dec, 4),
    )


# ============================================================
# API pública
# ============================================================
def visualize_logistic(
    trained_estimator,
    X, y,
    *,
    steps=60,
    mode="auto",
    show_loss=True,
    title=None,
    smooth="ema",
    smooth_beta=0.85,
    strict_loss=False,
    baseline="prior",
    display_space="original",
    dec=4,
):
    hist = fit_history_logistic(
        trained_estimator, X, y,
        steps=steps,
        mode=mode,
        smooth=smooth,
        smooth_beta=smooth_beta,
        baseline=baseline,
        display_space=display_space,
    )

    return build_logistic_figure(
        X, y,
        history=hist,
        show_loss=show_loss,
        title=title,
        strict_loss=strict_loss,
        dec=dec,
    )

# Test D = 1

In [8]:
import numpy as np
from sklearn.linear_model import SGDClassifier

np.random.seed(7)
n = 180

scale_x = 0.45
x = scale_x * np.random.normal(0, 1.0, size=n)

true_w = 3.0
true_b = -0.2

def sigmoid(z):
    z = np.clip(z, -60, 60)
    return 1.0 / (1.0 + np.exp(-z))

p_true = sigmoid(true_w * x + true_b)
y = (np.random.rand(n) < p_true).astype(int)

X = x.reshape(-1, 1)

model = SGDClassifier(
    loss="log_loss",
    penalty=None,
    alpha=0.0,
    learning_rate="constant",
    eta0=0.06,
    shuffle=False,
    max_iter=2000,
    tol=1e-6,
    random_state=7,
)

model.fit(X, y)

fig = visualize_logistic(
    model, X, y,
    steps=80,
    mode="iterative",
    show_loss=False,
    title="Binary Logistic Regression (1 variable) — Slow/Smooth Synthetic Data",
    smooth="ema",
    smooth_beta=0.85,
    baseline="prior",
    display_space="original",
    dec=4,
)

fig.show()